# UNIVERSIDAD AUTONOMA DE AGUASCALIENTES
# Departamento: Ciencias de la Computación
# Carrera: Ingeniería en Computación Inteligente

## Curso: Machine y Deep Learning
## Maestro: Dr. Francisco Javier Luna Rosas
## Alumno: Guillermo González Lara


### Semestre: Enero-Junio del 2026

PRACTICA No. 28. Ejemplo de Mnist con Transfer Learning (Redes Neuronales
Convolucionales-CNN).
¿Qué pasaria si se pudiera tomar an modelo existente que está ya entrenado en muchos más datos y usar las caracterísicas que ese modelo aprendió para aplicarlo a nuestros datos?. Este es el concepto de Fransfer
Learning y es lo que exploraremos en esta practica.
Transfer Learning es una de las técnicas centraies actualmente en Dees Leaming. Su interés radica en que, en lugar de necesitar eatrenar una red neuronal desde cero, lo que impica disponer de una gran cantidad de catos y de mucho tiempo (días o semanas) de computación para enfrenar, lo hacemos desde una red pre-entrenada. Esta
¿écnica nos permite descargar un modelo de código abierto que alguien ya ha entrenado previamente en un gran conjunto de datos y usar sus parámetros (miles o miliones) como pento de partida para, posteriormente, continuar entrenando el modeio con el conjunto de datos (más pequeño) que tengamos para ana tarea determinada.
en concreto en le página de modelos pre-entrenados a keras apptications podemos encontrar disponibies los siguientes modelos para clasificación de imágenes entrenados con imageNes: Xception, VGG16, VGG19,
ResNet50, inception:V3, InceptionResNetV2, MobieNet, MobfieNelV2, DeaseNet, NASNer.

### from keras.application.vgg19 import VGG19
### include_top=false,  no incluye la red fully conected
### layer. 

In [1]:
import os
import tensorflow as tf

# Force dynamic memory allocation globally
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

gpus = tf.config.list_physical_devices('GPU')
print("Active GPUs:", gpus)

if gpus:
    for gpu in gpus:
        print("Memory Growth Enabled:", tf.config.experimental.get_memory_growth(gpu))

I0000 00:00:1778482464.628745   10041 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778482464.673492   10041 pjrt_api.cc:95] PJRT_Api is set for device type gpu
I0000 00:00:1778482464.678005   10041 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778482465.772745   10041 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Active GPUs: []


W0000 00:00:1778482466.050411   10108 cuda_executor.cc:1532] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1778482466.179884   10041 gpu_device.cc:2364] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


### Paso 1: Importar librerías necesarias

In [ ]:
import os 
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import tensorflow.keras as keras
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Dense,Dropout,Flatten, Activation
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.layers import LeakyReLU

from keras.applications.vgg19 import VGG19
from keras.applications.vgg19 import preprocess_input


### Paso 2: Cargamos el dataset

In [ ]:
from tensorflow.keras.utils import plot_model
from tensorflow.keras.datasets import mnist

#cargar dataset

(X_train, y_train), (X_test, y_test) = mnist.load_data()
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

#ejemplo de imagen
import tensorflow as tf

# Place operations on the GPU
with tf.device('/GPU:0'):
    a = tf.constant([[1.0, 2.0], [3.0, 4.0]])
    b = tf.constant([[1.0, 1.0], [0.0, 1.0]])
    c = tf.matmul(a, b)
    print(c)

In [ ]:
import tensorflow as tf

# This forces TF to print the device (CPU or GPU) for every operation
tf.debugging.set_log_device_placement(True)

# Create some tensors
a = tf.constant([[1.0, 2.0], [3.0, 4.0]])
b = tf.constant([[1.0, 1.0], [0.0, 1.0]])

# Run a calculation
c = tf.matmul(a, b)

print("Result:", c)

In [ ]:
import tensorflow as tf

# Grab the GPU instance
gpus = tf.config.list_physical_devices('GPU')
if gpus:
  try:
    # Force TensorFlow to allocate VRAM dynamically
    for gpu in gpus:
      tf.config.experimental.set_memory_growth(gpu, True)
    print("Memory growth enabled for:", gpus)
  except RuntimeError as e:
    # Visible devices must be set before GPUs have been initialized
    print(e)

### Paso 3: graficamos imagenes


In [ ]:
fig, ax = plt.subplots(8, 8, figsize=(6, 6), dpi=100)
for i, axi in enumerate(ax.flat):
    axi.imshow(X_train[i], cmap='binary', vmin=0, vmax=255 )
    axi.set(xticks=[], yticks=[])
    
fig, ax = plt.subplots(8, 8, figsize=(6, 6), dpi=100)
for i, axi in enumerate(ax.flat):
    axi.imshow(X_test[i], cmap='binary', vmin=0, vmax=255 )
    axi.set(xticks=[], yticks=[])
    

### Paso 4: Transformacion de datos


In [ ]:

#Calculamos la cantidad de etiquetas
num_labels= len(np.unique(y_train))
print( "Numero de etiquetas: ",num_labels)

#Convertimos las etiquetas a formato one-hot
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)
print(y_train[0:8], "\n")

#Dimensiones de entrada de la imagen
image_size = X_train.shape[1]
print("Tamaño de la imagen: ", image_size)

#Redimensionamos las imagenes para que sean compatibles con la entrada de la red
X_train=np.reshape(X_train, [-1,image_size, image_size, 1])
X_test=np.reshape(X_test, [-1,image_size, image_size, 1])
X_train = X_train.astype('float32') / 255
X_test = X_test.astype('float32') / 255

print("Dimensiones de X_train: ", X_train.shape)
print("Dimensiones de X_test: ", X_test.shape)





### Paso 5: Procesar datos para hacerlos compatibles con VGG19




In [ ]:
from tensorflow.keras.preprocessing import image

from keras.preprocessing.image import img_to_array , array_to_img

xtrain=np.dstack([X_train] * 3)
xtest=np.dstack([X_test] * 3)

#reshape
xtrain = xtrain.reshape(-1,28,28,3)
xtest = xtest.reshape(-1,28,28,3)
print("Reshape image as per the tensor format required", xtrain.shape, xtest.shape)

#resize images
x_train=np.asarray([img_to_array(array_to_img(im, scale=False).resize((32, 32))) for im in xtrain])
x_test=np.asarray([img_to_array(array_to_img(im, scale=False).resize((32, 32))) for im in xtest])

print("Reshape image to 32x32 as required for VGG19", x_train.shape, x_test.shape)

### paso 6: Normalizamos datos

In [ ]:
x_train = x_train.astype('float32') / 255
x_test = x_test.astype('float32') / 255

### Paso 7: Construimos la estructura de la red

In [ ]:
import matplotlib.pyplot as plt
input_shape = (image_size, image_size, 1)
kernel_size = 3 
pool_size = 2
dropout = 0.2
filters = 64
num_labels = 10

vgg19 = VGG19(weights='imagenet', include_top=False, input_shape=(32, 32, 3))

#Congelar las capas para evitar que los pesos se actualicen durante el entrenamiento
for layer in vgg19.layers:
    layer.trainable = False
    
vgg19.summary()

model = keras.Sequential()
model.add(vgg19)

#Aplanamos datos
model.add(Flatten())

# dropput added as regularizer
model.add(Dropout(dropout))
#Hidden layer
model.add(Dense(40, activation='relu'))
model.add(Dense(20,activation='sigmoid'))

#Capa de salida
model.add(Dense(num_labels))
model.add(Activation('softmax'))

#Resumen estructura
model.summary()

# Paso 8: Configuracion del modelo

In [ ]:
#Funcion de costo para hot-one-hot-vector
model.compile(loss='categorical_crossentropy', metrics=['accuracy'], optimizer='adam')



### Paso 9: Entrenamos modelo

In [ ]:
batch_size= 128
model.fit(x_train, y_train, batch_size=batch_size, epochs=10)

In [ ]:
# Carga el mnist dataset
train, (x_test, y_test) = mnist.load_data()

# Estos datos son los que puede procesar el modelo
x_test = np.dstack([x_test] * 3)
x_test = x_test.reshape(-1, 28, 28, 3)
x_test_model = np.asarray([img_to_array(array_to_img(im, scale=False).resize((32,32))) for im in x_test])
x_test_model = x_test_model.astype('float32') / 255
print(x_test_model.shape)

# Graficamos los numeros que vamos a intentar clasificar
fig, ax = plt.subplots(4, 4, figsize = (6, 6), dpi = 100)
for i, axi in enumerate(ax.flat):
    axi.imshow(x_test[i][:,:,0], cmap='binary', vmin=0, vmax=255)
    axi.set(xticks=[], yticks=[])

In [ ]:
# Predecimos 16 imágenes
pred = model.predict(x_test_model[0:16])

# Para obtener las "etiquetas"
pred = np.argmax(pred, axis = -1)

# Mostramos las predicciones
print("Predicciones en fila: ", pred, "\n")
print("Predicciones 4x4: \n", np.reshape(pred, [4,4]))

## Conclusiones 

En esta práctica implementamos un modelo CNN para la identificación de imágenes existentes en la libería keras llamada mnist, integrada en tensorflow.

### URL: ImageDataGenerator (2024) https://keras.io/search.html?query=Data%20aumentation